# Train Experimental Cascaded PhoWhisper

This notebook trains an experimental cascaded encoder variant of PhoWhisper in PyTorch. It is intended for Colab or Kaggle.

Important: this is a research/training notebook. The custom cascaded encoder is not directly convertible to faster-whisper/CTranslate2 without native runtime/converter work. Use the existing CTranslate2 PhoWhisper path for production inference while evaluating this model.

## 1. Install Dependencies

Run this once per Colab/Kaggle session. Restart the kernel if your environment asks for it.

In [ ]:
!pip install -q "torch" "torchaudio" "transformers>=4.40" "datasets" "accelerate" "evaluate" "jiwer" "librosa" "soundfile" "safetensors" "faster-whisper" "ctranslate2"

## 2. Runtime and Training Configuration

By default this notebook uses [`ademax/vivos-vie-speech2text`](https://huggingface.co/datasets/ademax/vivos-vie-speech2text), a Hugging Face version of the Vietnamese VIVOS speech-to-text dataset.

For Kaggle offline runs, add a VIVOS-style dataset as an input and set `custom_manifest_path` to a CSV with columns like `audio` and `transcription`.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class TrainConfig:
    model_id: str = "vinai/PhoWhisper-base"
    output_dir: str = "./cascaded_phowhisper_ckpt"
    language: str = "vi"
    task: str = "transcribe"
    max_new_tokens: int = 96

    # Dataset options. Default is ademax/vivos-vie-speech2text from Hugging Face.
    custom_manifest_path: str | None = None
    dataset_name: str = "ademax/vivos-vie-speech2text"
    dataset_config: str | None = None
    train_split: str = "train"
    eval_split: str = "test"
    text_column: str = "transcription"

    # Architecture. Replace early PhoWhisper encoder layers with a smaller frontend.
    refine_start_layer: int = 3
    fast_hidden_size: int = 256
    fast_num_layers: int = 2
    fast_num_heads: int = 4
    fast_ffn_dim: int = 1024
    lambda_distill: float = 0.3
    freeze_decoder_steps: int = 1000

    # Longer VIVOS training defaults. Lower max_steps for a quick smoke run.
    batch_size: int = 1
    grad_accum_steps: int = 16
    max_steps: int = 5000
    learning_rate: float = 1e-5
    warmup_steps: int = 300
    log_every: int = 10
    eval_every: int = 500
    save_every: int = 500
    # Keep this at 0 on Colab/Kaggle to avoid DataLoader worker RAM duplication.
    num_workers: int = 0
    max_audio_seconds: float = 30.0
    max_label_tokens: int = 224

    # Baseline comparison. Point this at your converted CTranslate2 PhoWhisper model.
    ct2_model_path: str = "./phowhisper-base-ct2"
    ct2_quantization: str = "float16"
    ct2_force_convert: bool = False
    compare_eval_samples: int = 50
    compare_compute_type: str = "float16"
    seed: int = 42

cfg = TrainConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
import math
import os
import random
import time
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import Audio, Dataset, DatasetDict, load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup
from transformers.models.whisper.modeling_whisper import shift_tokens_right

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("device:", device)
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

## 3. Load Dataset

Expected custom manifest columns:

- `audio`, `audio_path`, `path`, or `file`
- `transcription`, `text`, `sentence`, `transcript`, or `raw_transcription`

`ademax/vivos-vie-speech2text` provides `audio`, `transcription`, and `raw_transcription`. The notebook uses `transcription` because it keeps readable Vietnamese casing and punctuation. Audio is resampled to 16 kHz by `datasets.Audio`.

In [ ]:
def first_existing_column(columns, candidates):
    for name in candidates:
        if name in columns:
            return name
    raise ValueError(f"None of {candidates} found in columns: {columns}")

def load_asr_dataset(cfg: TrainConfig):
    if cfg.custom_manifest_path:
        data = load_dataset("csv", data_files={"train": cfg.custom_manifest_path})
        data = DatasetDict({"train": data["train"], "validation": data["train"].select(range(min(100, len(data["train"]))))})
    else:
        dataset_kwargs = {"split": cfg.train_split}
        eval_kwargs = {"split": cfg.eval_split}
        if cfg.dataset_config:
            train_data = load_dataset(cfg.dataset_name, cfg.dataset_config, **dataset_kwargs)
            eval_data = load_dataset(cfg.dataset_name, cfg.dataset_config, **eval_kwargs)
        else:
            train_data = load_dataset(cfg.dataset_name, **dataset_kwargs)
            eval_data = load_dataset(cfg.dataset_name, **eval_kwargs)
        data = DatasetDict({"train": train_data, "validation": eval_data})

    audio_col = first_existing_column(data["train"].column_names, ["audio", "audio_path", "path", "file"])
    text_col = cfg.text_column if cfg.text_column in data["train"].column_names else first_existing_column(
        data["train"].column_names,
        ["transcription", "text", "sentence", "transcript", "raw_transcription"],
    )

    if audio_col != "audio":
        data = data.rename_column(audio_col, "audio")
    if text_col != "text":
        data = data.rename_column(text_col, "text")

    data = data.cast_column("audio", Audio(sampling_rate=16000))
    return data

raw_data = load_asr_dataset(cfg)
print(raw_data)
print(raw_data["train"][0].keys())

## 4. Lazy Preprocessing and Batch Collation

Do **not** precompute log-Mel features for the full VIVOS dataset on Colab. That can fill system RAM. This notebook computes audio features lazily inside the DataLoader collator, so only the current mini-batch is held in memory.

In [ ]:
processor = WhisperProcessor.from_pretrained(cfg.model_id, language=cfg.language, task=cfg.task)

# Keep the Hugging Face dataset in its compact Arrow/audio form. Features and
# token labels are created lazily per batch in WhisperDataCollator below.
print(raw_data)
print("train columns:", raw_data["train"].column_names)
print("validation columns:", raw_data["validation"].column_names)

In [ ]:
class WhisperDataCollator:
    def __init__(self, processor: WhisperProcessor):
        self.processor = processor

    def __call__(self, features):
        audio_arrays = []
        transcripts = []

        for item in features:
            audio = item["audio"]
            audio_array = audio["array"]
            duration = len(audio_array) / audio["sampling_rate"]
            if duration <= cfg.max_audio_seconds:
                audio_arrays.append(audio_array)
                transcripts.append(item["text"])

        if not audio_arrays:
            audio = features[0]["audio"]
            audio_arrays = [audio["array"][: int(cfg.max_audio_seconds * audio["sampling_rate"])]]
            transcripts = [features[0]["text"]]

        batch = self.processor.feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            return_tensors="pt",
        )

        label_features = self.processor.tokenizer(
            transcripts,
            truncation=True,
            max_length=cfg.max_label_tokens,
        )
        labels_batch = self.processor.tokenizer.pad(
            [{"input_ids": ids} for ids in label_features.input_ids],
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch

collator = WhisperDataCollator(processor)
train_loader = DataLoader(
    raw_data["train"],
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collator,
    num_workers=cfg.num_workers,
)
eval_loader = DataLoader(
    raw_data["validation"],
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collator,
    num_workers=cfg.num_workers,
)
batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in batch.items()}

## 5. Cascaded PhoWhisper Model

The student reuses PhoWhisper's convolutional frontend, positional embeddings, encoder layers, decoder, and output projection. The encoder layers are split into a fast stage and a refinement stage.

This is a practical prototype. For a truly faster architecture, replace the fast stage or refinement stage with a smaller module, then distill it against the teacher encoder.

In [ ]:
class LightweightFastEncoderStage(nn.Module):
    """Small low-width acoustic frontend that replaces early PhoWhisper layers."""

    def __init__(self, config, fast_hidden_size: int, fast_num_layers: int, fast_num_heads: int, fast_ffn_dim: int):
        super().__init__()
        num_mel_bins = getattr(config, "num_mel_bins", 80)
        self.conv1 = nn.Conv1d(num_mel_bins, fast_hidden_size, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(fast_hidden_size, fast_hidden_size, kernel_size=3, stride=2, padding=1)
        self.embed_positions = nn.Embedding(config.max_source_positions, fast_hidden_size)
        self.dropout = config.dropout
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=fast_hidden_size,
            nhead=fast_num_heads,
            dim_feedforward=fast_ffn_dim,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=fast_num_layers)
        self.fast_norm = nn.LayerNorm(fast_hidden_size)
        self.to_phowhisper_width = nn.Linear(fast_hidden_size, config.d_model)

    def forward(self, input_features: torch.Tensor) -> torch.Tensor:
        hidden_states = F.gelu(self.conv1(input_features))
        hidden_states = F.gelu(self.conv2(hidden_states))
        hidden_states = hidden_states.permute(0, 2, 1)

        positions = self.embed_positions.weight[: hidden_states.size(1)]
        hidden_states = hidden_states + positions.unsqueeze(0).to(hidden_states.dtype)
        hidden_states = F.dropout(hidden_states, p=self.dropout, training=self.training)
        hidden_states = self.layers(hidden_states)
        hidden_states = self.fast_norm(hidden_states)
        return self.to_phowhisper_width(hidden_states)

class WhisperRefineEncoderStage(nn.Module):
    def __init__(self, whisper_encoder: nn.Module, refine_start_layer: int):
        super().__init__()
        self.embed_positions = whisper_encoder.embed_positions
        self.layers = nn.ModuleList(list(whisper_encoder.layers[refine_start_layer:]))
        self.layer_norm = whisper_encoder.layer_norm

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        positions = self.embed_positions.weight[: hidden_states.size(1)]
        hidden_states = hidden_states + positions.unsqueeze(0).to(hidden_states.dtype)
        for layer in self.layers:
            layer_outputs = layer(hidden_states, attention_mask=None)
            hidden_states = layer_outputs[0] if isinstance(layer_outputs, tuple) else layer_outputs
        return self.layer_norm(hidden_states)

class CascadedEncoder(nn.Module):
    def __init__(self, fast_encoder: nn.Module, refine_encoder: nn.Module):
        super().__init__()
        self.fast_encoder = fast_encoder
        self.refine_encoder = refine_encoder

    def forward(self, input_features: torch.Tensor) -> torch.Tensor:
        fast_features = self.fast_encoder(input_features)
        refined_features = self.refine_encoder(fast_features)
        return refined_features

class CascadedPhoWhisperForConditionalGeneration(nn.Module):
    def __init__(self, base_model: WhisperForConditionalGeneration, cfg: TrainConfig):
        super().__init__()
        self.config = base_model.config
        encoder = base_model.model.encoder
        self.encoder = CascadedEncoder(
            LightweightFastEncoderStage(
                self.config,
                cfg.fast_hidden_size,
                cfg.fast_num_layers,
                cfg.fast_num_heads,
                cfg.fast_ffn_dim,
            ),
            WhisperRefineEncoderStage(encoder, cfg.refine_start_layer),
        )
        self.decoder = base_model.model.decoder
        self.proj_out = base_model.proj_out

    def forward(self, input_features, labels=None, decoder_input_ids=None):
        encoder_hidden_states = self.encoder(input_features)

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError("decoder_input_ids or labels must be provided")
            decoder_input_ids = shift_tokens_right(
                labels,
                self.config.pad_token_id,
                self.config.decoder_start_token_id,
            )
        decoder_attention_mask = decoder_input_ids.ne(self.config.pad_token_id).long()
        decoder_outputs = self.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            use_cache=False,
            return_dict=True,
        )
        logits = self.proj_out(decoder_outputs.last_hidden_state)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                ignore_index=-100,
            )

        return {"loss": loss, "logits": logits, "encoder_hidden_states": encoder_hidden_states}

    @torch.no_grad()
    def generate_greedy(self, input_features, processor, max_new_tokens=96):
        self.eval()
        prompt = [self.config.decoder_start_token_id]
        for _, token_id in processor.get_decoder_prompt_ids(language=cfg.language, task=cfg.task):
            prompt.append(token_id)
        decoder_input_ids = torch.tensor([prompt], device=input_features.device)

        for _ in range(max_new_tokens):
            out = self(input_features, decoder_input_ids=decoder_input_ids)
            next_token = out["logits"][:, -1].argmax(dim=-1, keepdim=True)
            decoder_input_ids = torch.cat([decoder_input_ids, next_token], dim=-1)
            if next_token.item() == self.config.eos_token_id:
                break
        return decoder_input_ids

In [ ]:
teacher = WhisperForConditionalGeneration.from_pretrained(
    cfg.model_id,
    attn_implementation="eager",
).to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

student_base = WhisperForConditionalGeneration.from_pretrained(
    cfg.model_id,
    attn_implementation="eager",
)
student = CascadedPhoWhisperForConditionalGeneration(student_base, cfg).to(device)

print("teacher params:", sum(p.numel() for p in teacher.parameters()) / 1e6, "M")
print("student params:", sum(p.numel() for p in student.parameters()) / 1e6, "M")
print("refine starts at PhoWhisper encoder layer:", cfg.refine_start_layer)
print("fast encoder hidden/layers/heads:", cfg.fast_hidden_size, cfg.fast_num_layers, cfg.fast_num_heads)

## 6. Train with ASR Loss + Encoder Distillation

Loss:

`loss = cross_entropy(decoder(student_encoder(mel)), labels) + lambda_distill * mse(student_encoder(mel), teacher_encoder(mel))`

The first `freeze_decoder_steps` train only the cascaded encoder so the refined features learn to imitate the teacher encoder before decoder fine-tuning.

In [ ]:
def set_decoder_trainable(model, trainable: bool):
    for p in model.decoder.parameters():
        p.requires_grad = trainable
    for p in model.proj_out.parameters():
        p.requires_grad = trainable

set_decoder_trainable(student, False)

optimizer = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=cfg.learning_rate)
total_optimizer_steps = cfg.max_steps
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg.warmup_steps,
    num_training_steps=total_optimizer_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def save_checkpoint(step: int):
    out_dir = Path(cfg.output_dir) / f"step_{step}"
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "step": step,
            "model_state_dict": student.state_dict(),
            "model_id": cfg.model_id,
            "language": cfg.language,
            "task": cfg.task,
            "max_new_tokens": cfg.max_new_tokens,
            "refine_start_layer": cfg.refine_start_layer,
            "fast_hidden_size": cfg.fast_hidden_size,
            "fast_num_layers": cfg.fast_num_layers,
            "fast_num_heads": cfg.fast_num_heads,
            "fast_ffn_dim": cfg.fast_ffn_dim,
            "lambda_distill": cfg.lambda_distill,
        },
        out_dir / "cascaded_phowhisper.pt",
    )
    processor.save_pretrained(out_dir / "processor")
    student.config.save_pretrained(out_dir / "config")
    print(f"saved checkpoint: {out_dir}")

@torch.no_grad()
def evaluate_loss(max_batches=20):
    student.eval()
    losses = []
    for i, batch in enumerate(eval_loader):
        if i >= max_batches:
            break
        input_features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)
        out = student(input_features=input_features, labels=labels)
        losses.append(out["loss"].item())
    student.train()
    return float(np.mean(losses)) if losses else math.nan

In [ ]:
student.train()
global_step = 0
running = {"loss": 0.0, "asr": 0.0, "distill": 0.0}
started = time.time()

while global_step < cfg.max_steps:
    for batch in train_loader:
        if global_step == cfg.freeze_decoder_steps:
            print("unfreezing decoder")
            set_decoder_trainable(student, True)
            optimizer = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=cfg.learning_rate)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=max(1, cfg.warmup_steps // 2),
                num_training_steps=max(1, cfg.max_steps - global_step),
            )

        input_features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            out = student(input_features=input_features, labels=labels)
            asr_loss = out["loss"]
            with torch.no_grad():
                teacher_hidden = teacher.model.encoder(input_features=input_features).last_hidden_state
            distill_loss = F.mse_loss(out["encoder_hidden_states"].float(), teacher_hidden.float())
            loss = asr_loss + cfg.lambda_distill * distill_loss
            scaled_loss = loss / cfg.grad_accum_steps

        scaler.scale(scaled_loss).backward()

        if (global_step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running["loss"] += loss.item()
        running["asr"] += asr_loss.item()
        running["distill"] += distill_loss.item()
        global_step += 1

        if global_step % cfg.log_every == 0:
            denom = cfg.log_every
            elapsed = time.time() - started
            lr = scheduler.get_last_lr()[0]
            print(
                f"step={global_step} loss={running['loss']/denom:.4f} "
                f"asr={running['asr']/denom:.4f} distill={running['distill']/denom:.4f} "
                f"lr={lr:.2e} elapsed={elapsed/60:.1f}m"
            )
            running = {"loss": 0.0, "asr": 0.0, "distill": 0.0}

        if global_step % cfg.eval_every == 0:
            print("eval_loss:", evaluate_loss(max_batches=20))

        if global_step % cfg.save_every == 0:
            save_checkpoint(global_step)

        if global_step >= cfg.max_steps:
            break

save_checkpoint(global_step)

## 7. Quick Inference Smoke Test

This greedy decoder is intentionally simple. Use it only as a smoke test. For real evaluation, compute WER/CER over the validation set and compare with the original PhoWhisper model and the faster-whisper converted baseline.

In [ ]:
sample = raw_data["validation"][0]
sample_audio = sample["audio"]
sample_features = processor.feature_extractor(
    sample_audio["array"],
    sampling_rate=sample_audio["sampling_rate"],
    return_tensors="pt",
).input_features
input_features = sample_features.to(device)
with torch.no_grad():
    pred_ids = student.generate_greedy(input_features, processor, max_new_tokens=cfg.max_new_tokens)
text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
print("prediction:", text)
print("label:", sample["text"])

## 8. Convert PhoWhisper to CTranslate2

Run this before the faster-whisper comparison if `cfg.ct2_model_path` does not already contain a converted CTranslate2 model. This converts the standard PhoWhisper baseline, not the custom cascaded PyTorch encoder.

In [ ]:
import subprocess
import sys

ct2_path = Path(cfg.ct2_model_path)
model_bin = ct2_path / "model.bin"

if model_bin.exists() and not cfg.ct2_force_convert:
    print(f"CTranslate2 model already exists: {ct2_path}")
else:
    ct2_path.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "-m",
        "ctranslate2.converters.transformers",
        "--model",
        cfg.model_id,
        "--output_dir",
        str(ct2_path),
        "--quantization",
        cfg.ct2_quantization,
        "--copy_files",
        "tokenizer.json",
        "preprocessor_config.json",
        "--force",
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
    print(f"Converted CTranslate2 model written to: {ct2_path}")

print("CT2 files:", sorted(p.name for p in ct2_path.glob("*")))

## 9. Compare Against faster-whisper / CTranslate2 PhoWhisper

This compares the experimental PyTorch cascade against the converted CTranslate2 PhoWhisper baseline on the same validation samples.

In [ ]:
import tempfile
import jiwer
import soundfile as sf
from faster_whisper import WhisperModel

def normalize_vi_text(text: str) -> str:
    text = (text or "").lower().strip()
    text = " ".join(text.split())
    return text

def transcribe_cascade_sample(sample, max_new_tokens=cfg.max_new_tokens):
    audio = sample["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    features = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    ).input_features.to(device)
    started = time.perf_counter()
    with torch.no_grad():
        pred_ids = student.generate_greedy(features, processor, max_new_tokens=max_new_tokens)
    seconds = time.perf_counter() - started
    text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
    return text, seconds, duration

def transcribe_faster_whisper_sample(model, sample):
    audio = sample["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    with tempfile.NamedTemporaryFile(suffix=".wav") as tmp:
        sf.write(tmp.name, audio["array"], audio["sampling_rate"])
        started = time.perf_counter()
        segments, _ = model.transcribe(
            tmp.name,
            language=cfg.language,
            task=cfg.task,
            beam_size=3,
            vad_filter=False,
        )
        text = " ".join(seg.text.strip() for seg in segments).strip()
        seconds = time.perf_counter() - started
    return text, seconds, duration

def summarize_asr_metrics(rows, name):
    refs = [normalize_vi_text(r["reference"]) for r in rows]
    hyps = [normalize_vi_text(r[name]) for r in rows]
    total_seconds = sum(r[f"{name}_seconds"] for r in rows)
    total_audio = sum(r["audio_seconds"] for r in rows)
    return {
        "wer": jiwer.wer(refs, hyps),
        "cer": jiwer.cer(refs, hyps),
        "rtf": total_seconds / total_audio if total_audio else float("nan"),
        "decode_seconds": total_seconds,
        "audio_seconds": total_audio,
    }

ct2_path = Path(cfg.ct2_model_path)
if not ct2_path.exists():
    raise FileNotFoundError(
        f"Converted CTranslate2 model not found: {ct2_path}. "
        "Set cfg.ct2_model_path to your converted PhoWhisper CT2 directory."
    )

fw_device = "cuda" if device == "cuda" else "cpu"
fw_compute_type = cfg.compare_compute_type if fw_device == "cuda" else "int8"
fw_model = WhisperModel(str(ct2_path), device=fw_device, compute_type=fw_compute_type)

student.eval()
rows = []
for idx in range(min(cfg.compare_eval_samples, len(raw_data["validation"]))):
    sample = raw_data["validation"][idx]
    reference = sample["text"]
    cascade_text, cascade_seconds, audio_seconds = transcribe_cascade_sample(sample)
    fw_text, fw_seconds, _ = transcribe_faster_whisper_sample(fw_model, sample)
    rows.append({
        "reference": reference,
        "cascade": cascade_text,
        "cascade_seconds": cascade_seconds,
        "faster_whisper": fw_text,
        "faster_whisper_seconds": fw_seconds,
        "audio_seconds": audio_seconds,
    })
    if (idx + 1) % 10 == 0:
        print(f"evaluated {idx + 1}/{cfg.compare_eval_samples}")

cascade_metrics = summarize_asr_metrics(rows, "cascade")
fw_metrics = summarize_asr_metrics(rows, "faster_whisper")
print("cascade:", cascade_metrics)
print("faster_whisper:", fw_metrics)
print("sample row:", rows[0] if rows else None)

## 10. Next Steps

1. Tune `fast_hidden_size`, `fast_num_layers`, and `refine_start_layer` for your latency/accuracy target.
2. Train longer on the full in-domain VIVOS data or your own production Vietnamese audio.
3. Keep the custom cascade in PyTorch/ONNX. Use faster-whisper/CTranslate2 as the production baseline unless you intentionally implement native CTranslate2 support for the custom encoder.